<br>

<center><img src='../Header_image/header_image.png' style="width:900px"></center>

<br>

<p style="font-size: 30px; text-align: center;"> 
<b>✏️Homework Session 8 - 2.5D DC Resistivity Inversion</b></p>

___


The objective of this homework is to perform a 2.5D inversion of DC resistivity data from simulated data from a simple 2D model. The task consists of inverting using the following regularization methods:
- Weigthed Least Squares $\rightarrow$ smooth model
- Iteratively Reweighted Least Squares $\rightarrow$ blocky model

We will use synthetic data generated from a single 2D model, and we will use the same mesh and data for both inversions.

In [5]:
# SimPEG functionality
from simpeg.electromagnetics.static import resistivity as dc
from simpeg.electromagnetics.static.utils.static_utils import (
    plot_pseudosection,
    generate_survey_from_abmn_locations,
    apparent_resistivity_from_voltage,
)
from simpeg.utils.io_utils.io_utils_electromagnetics import read_dcip2d_ubc
from simpeg.utils import download, model_builder
from simpeg import (
    maps,
    data,
    data_misfit,
    regularization,
    optimization,
    inverse_problem,
    inversion,
    directives,
)

# discretize functionality
from discretize import TreeMesh
from discretize.utils import active_from_xyz

# Basic Python functionality
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import tarfile

mpl.rcParams.update({"font.size": 14})  # default font size
cmap = mpl.cm.RdYlBu_r  # default colormap

### Plot true conductivity model

The synthetic data is generated from this simple 2D model, which consists of a background conductivity of 0.01 S/m a resistor with conductivity of 0.001 S/m. The resistor is located at (120 m, 72 m) and has a radius of 60 m. Similarly, there is a conductive anomaly with conductivity of 0.1 S/m located at (240 m, 40 m) with a radius of 60 m. The topography is also included in the model, which is defined by a sinusoidal function.

The true conductivity model is shown below:
<img src="./NB_images/homework_true_model.png" />

### Load the data

In [ ]:
# path to the directory containing our data
dir_path = ".\data"

# files to work with
topo_filename = os.path.join(dir_path, "topo_2d.txt")
data_filename = os.path.join(dir_path, "dc_data.obs")

<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
C:\Users\u_ber\AppData\Local\Temp\ipykernel_3148\479711034.py:2: SyntaxWarning: invalid escape sequence '\d'
  dir_path = ".\data"


The format for this DC resistivity data is as follows:

```
Comment line: description of data
   n_sources dipole(1) dipole(1) 
     A_location  B_location   n_receivers
   M_location  N_location  measured_potential  standard_deviation
   M_location  N_location  measured_potential  standard_deviation
     A_location  B_location   n_receivers
   M_location  N_location  measured_potential  standard_deviation
   M_location  N_location  measured_potential  standard_deviation
   ....
```

In [ ]:
with open(data_filename) as f:
    print(f.read())

COMMON_CURRENT
! general FORMAT
18
-4.000000e+02 1.402466e+02 -3.600000e+02 1.340021e+02 10
-3.200000e+02 1.283790e+02 -2.800000e+02 1.239188e+02 -1.546672e-01 7.855120e-03
-2.800000e+02 1.239188e+02 -2.400000e+02 1.210514e+02 -4.061362e-02 1.860216e-03
-2.400000e+02 1.210514e+02 -2.000000e+02 1.200525e+02 -1.635626e-02 7.936469e-04
-2.000000e+02 1.200525e+02 -1.600000e+02 1.210161e+02 -8.180299e-03 3.840065e-04
-1.600000e+02 1.210161e+02 -1.200000e+02 1.238450e+02 -2.327642e-03 1.217918e-04
-1.200000e+02 1.238450e+02 -8.000000e+01 1.282600e+02 -7.029581e-04 3.551757e-05
-8.000000e+01 1.282600e+02 -4.000000e+01 1.338272e+02 -7.249909e-04 3.386383e-05
-4.000000e+01 1.338272e+02 0.000000e+00 1.400000e+02 -5.134538e-04 2.832406e-05
0.000000e+00 1.400000e+02 4.000000e+01 1.461728e+02 -4.885169e-04 2.485250e-05
4.000000e+01 1.461728e+02 8.000000e+01 1.517400e+02 -4.335660e-04 2.129087e-05

-3.600000e+02 1.340021e+02 -3.200000e+02 1.283790e+02 10
-2.800000e+02 1.239188e+02 -2.400000e+02 1.21

<div class="alert alert-block alert-warning">
<font size="6"> &#128675;</font> <b>Exercise 1: </b>

Now that the data is generated, we will perform the following steps:
* Load the Topography into a `numpy` 2d array.
* Create DC voltage object with the `read_dcip2d_ubc` function.
* Assign reasonable uncertainties to the voltage object (voltage_object.standard_deviation = ...). You can use, for example, 5% of the measured potential as the standard deviation.
* Create a survey object using the `generate_survey_from_abmn_locations` function:
    
    survey = generate_survey_from_abmn_locations(
        locations_a=A,
        locations_b=B,
        locations_m=M,
        locations_n=N,
        data_type='volt'
)

where A, B, M, N are the locations of the electrodes in the survey extracted from the DC voltage object (A = voltage_object.survey.locations_a, ...).

</div>

### Design a (Tree) Mesh

Assuming your topography is called as `todo_2d` and your voltage object is called `voltage_data`, you can design a three mesh using the code below. A tree mesh is a type of mesh that allows for local refinement, which can be useful for resolving small-scale features in the model. In this example the mesh is refined around the locations of the electrodes and the topography.

```python

In [ ]:
dh = 4  # base cell width
dom_width_x = 3200.0  # domain width x
dom_width_z = 2400.0  # domain width z
nbcx = 2 ** int(np.round(np.log(dom_width_x / dh) / np.log(2.0)))  # num. base cells x
nbcz = 2 ** int(np.round(np.log(dom_width_z / dh) / np.log(2.0)))  # num. base cells z

# Define the base mesh with top at z = 0 m
hx = [(dh, nbcx)]
hz = [(dh, nbcz)]
mesh = TreeMesh([hx, hz], x0="CN", diagonal_balance=True)

# Shift top to maximum topography
mesh.origin = mesh.origin + np.r_[0.0, topo_2d[:, -1].max()]

# Mesh refinement based on topography
mesh.refine_surface(
    topo_2d,
    padding_cells_by_level=[0, 0, 4, 4],
    finalize=False,
)

# Extract unique electrode locations.
unique_locations = voltage_data.survey.unique_electrode_locations

# Mesh refinement near electrodes.
mesh.refine_points(
    unique_locations, padding_cells_by_level=[8, 12, 6, 6], finalize=False
)

mesh.finalize()

Optionally plot the topography and the survey locations to check that they are correctly loaded. The following cell plot shows the tree mesh design.

In [ ]:
# # Plot 2D topography
# fig = plt.figure(figsize=(10, 2))
# ax = fig.add_axes([0.1, 0.1, 0.8, 0.8])
# ax.plot(topo_2d[:, 0], topo_2d[:, -1], color="b", linewidth=1)
# ax.scatter(unique_locations[:, 0], unique_locations[:, 1], 8, "r", label="Electrodes")
# ax.set_xlim([topo_2d[:, 0].min(), topo_2d[:, 0].max()])
# ax.set_xlabel("x (m)", labelpad=5)
# ax.set_ylabel("z (m)", labelpad=5)
# ax.grid(True)
# ax.set_title("Topography (Exaggerated z-axis)", fontsize=16, pad=10)
# plt.legend(loc="upper right")
# plt.show(fig)

In [ ]:
# fig = plt.figure(figsize=(10, 4))

# ax1 = fig.add_axes([0.14, 0.17, 0.8, 0.7])
# mesh.plot_grid(ax=ax1, linewidth=1)
# ax1.grid(False)
# ax1.set_xlim(-1500, 1500)
# ax1.set_ylim(np.max(topo_2d[:, -1]) - 1000, np.max(topo_2d[:, -1]))
# ax1.set_title("Mesh")
# ax1.set_xlabel("x (m)")
# ax1.set_ylabel("z (m)")

# plt.show()

### Define the Active Cells

Simulated geophysical data are dependent on the subsurface distribution of physical property values. As a result, the cells lying below the surface topography are commonly referred to as 'active cells'. And air cells, whose physical property values are fixed, are commonly referred to as 'inactive cells'. Here, the discretize [active_from_xyz](xref:discretize#discretize.utils.active_from_xyz) utility function is used to find the indices of the active cells using the mesh and surface topography. The output quantity is a ``bool`` array.

In [ ]:
# Indices of the active mesh cells from topography (e.g. cells below surface)
active_cells = active_from_xyz(mesh, topo_2d)

# number of active cells
n_active = np.sum(active_cells)

voltage_data.survey.drape_electrodes_on_topography(
    mesh, active_cells, topo_cell_cutoff="top"
)

### Create initial/background model

In [ ]:
## Select an appropiate initial/background model
sig0 = 1 / np.median(apparent_resistivities)
# sig0 = 10**np.mean(np.log10(apparent_resistivities))
sig0

np.float64(0.010239354604786283)

In [ ]:
# Create background model
m0 = np.ones(n_active) * np.log(sig0)

Now that the data is generated, we will perform the following steps:
* Load the Topography into a `numpy` 2d array.
* Create DC voltage object with the `read_dcip2d_ubc` function.
* Assign reasonable uncertainties to the voltage object (voltage_object.standard_deviation = ...). You can use, for example, 5% of the measured potential as the standard deviation.
* Create a survey object using the `generate_survey_from_abmn_locations` function:
    
    survey = generate_survey_from_abmn_locations(
        locations_a=A,
        locations_b=B,
        locations_m=M,
        locations_n=N,
        data_type='volt'
)

where A, B, M, N are the locations of the electrodes in the survey extracted from the DC voltage object (A = voltage_object.survey.locations_a, ...).

</div>

<div class="alert alert-block alert-warning">
<font size="6"> &#128675;</font> <b>Exercise 2: </b>

#### Weighted Least-Squares Inversion
Steps to follow:
1. Construct forward simulation object. Requires: `mesh`, `mapping`, and `survey`.
2. Define data misfit function. Use the [DataMisfit.l2_DataMisfit](xref:simpeg#simpeg.data_misfit.l2_DataMisfit) class, which implements the L2 norm for the data misfit. Requires: `simulation` and `data`.
3. Define regularization object. You can use the
[Regularization.Simple](xref:simpeg#simpeg.regularization.Simple) class, which implements a Tikhonov regularization. Requires: `mesh` and `active_cells`.
4. Define the optimization method.
5. Define the inverse problem. Requires: `data_misfit`, `regularization`, and `optimize`.
6. *(Optional)* Define directives to update the inversion.
7. Run the inversion.

The result should look something like this:

<img src="./NB_images/homework_smooth_inv.png" />

<div class="alert alert-block alert-warning">
<font size="6"> &#128675;</font> <b>Exercise 3: </b>

#### Iteratively Re-weighted Least-Squares Inversion

Same steps to follow, but change step 3 to define the regularization object. Use the [Regularization.Sparse](xref:simpeg#simpeg.regularization.Sparse) class, which implements IRLS regularization. 

This inversion should produce a more blocky model like the following:


The result should look something like this:

<img src="./NB_images/homework_blocky_inv.png" />